# Qwen3-4B Compound Compression Energy Ablation

Reproducible Google Colab experiment for CSE407 Green Computing.

**Primary model:** `Qwen/Qwen3-4B` and `Qwen/Qwen3-4B-AWQ`  
**Hardware:** one NVIDIA T4 16 GB  
**Primary metrics:** GPU energy, power, latency, throughput, VRAM, NIAH retrieval  
**Thinking subset:** exact-answer arithmetic and logic at 8192 input tokens

Run sections in order. The notebook writes every completed run to Google Drive and
can resume after a Colab disconnect. It never silently relabels a fallback as the
requested method.

## Important implementation status

- C0 and C1 use Hugging Face full-precision dynamic KV caches. The T4 has no
  native BF16 support, so C0 uses FP16 and is reported as the full-precision
  baseline. Calling a T4 run "BF16" would be technically misleading.
- C2 first attempts the SHA-pinned community TurboQuant cache. It is accepted only
  after shape, packing, sequence-length, and physical-storage checks. If it fails,
  the actual method is labeled `HF_HQQ_4BIT`, a KIVI-inspired fallback.
- Official KVCache-Factory PyramidKV and standalone SnapKV do **not** support Qwen3.
  They patch Llama/Mistral classes and require incompatible Transformers versions.
  C3/C4 therefore fail closed unless a separately validated Qwen3 adapter is supplied.
  FullKV duplicates are not recorded as eviction results.

This status is intentional research provenance, not a notebook error.

## 1. Install pinned dependencies

Run this cell once, then use **Runtime > Restart session**, and continue at Section 2.
AutoAWQ is deprecated and may downgrade Transformers, so Transformers 5.3.0 is
reinstalled last. The exact TurboQuant community commit is pinned.

In [ ]:
%pip install -q "autoawq==0.2.9" "hqq==0.2.8.post1" "nvidia-ml-py==12.575.51" \
    "pandas==2.3.3" "scipy==1.16.3" "statsmodels==0.14.5" \
    "matplotlib==3.10.7" "seaborn==0.13.2" "accelerate==1.11.0"
%pip install -q --upgrade "transformers==5.3.0"
%pip install -q --no-deps \
    "git+https://github.com/back2matching/turboquant.git@4de496d48bff9a7524016f60396aa374a999c887"
print("Restart the Colab session before continuing.")

## 2. Imports, Drive, configuration, and environment manifest

In [ ]:
from __future__ import annotations

import contextlib
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import subprocess
import threading
import time
import traceback
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st
import seaborn as sns
import torch
import transformers
from scipy.stats import binomtest, friedmanchisquare, wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache, QuantizedCache

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_ROOT = Path("/content/drive/MyDrive/CSE407_Qwen3_Energy")
except ImportError:
    DEFAULT_ROOT = Path.cwd() / "qwen3_energy_results"

ROOT = Path(os.environ.get("CSE407_RESULTS_DIR", DEFAULT_ROOT))
RAW_DIR = ROOT / "raw"
TRACE_DIR = ROOT / "power_traces"
FIG_DIR = ROOT / "figures"
for directory in (ROOT, RAW_DIR, TRACE_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 407
CONTEXT_LENGTHS = (512, 2048, 8192)
PERFORMANCE_BLOCKS = 10
NIAH_ITEMS_PER_CONDITION = 30
REASONING_ITEMS = 30
DECODE_TOKENS = 128
NIAH_MAX_NEW_TOKENS = 24
THINKING_MAX_NEW_TOKENS = 2048
POLL_INTERVAL_S = 0.1
COOLDOWN_TARGET_C = 72
COOLDOWN_TIMEOUT_S = 120
MODEL_BF16 = "Qwen/Qwen3-4B"
MODEL_AWQ = "Qwen/Qwen3-4B-AWQ"
TURBOQUANT_COMMIT = "4de496d48bff9a7524016f60396aa374a999c887"
RUN_FULL_PERFORMANCE = False
RUN_FULL_NIAH = False
RUN_FULL_THINKING = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULT_COLUMNS = [
    "run_id", "timestamp_utc", "experiment", "block", "item_id", "config",
    "requested_method", "actual_method", "method_validated", "model_id",
    "context_target", "input_tokens", "output_tokens", "thinking",
    "needle_depth", "correct", "response", "prefill_s", "decode_s", "total_s",
    "prefill_tok_s", "decode_tok_s", "avg_power_w", "min_power_w", "max_power_w",
    "gross_energy_j", "idle_power_w", "active_energy_j", "energy_per_output_token_j",
    "torch_loaded_mb", "torch_peak_mb", "nvml_peak_used_mb", "cache_live_mb",
    "power_samples", "trace_file", "valid", "error",
]

def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def gpu_manifest() -> dict[str, Any]:
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required. Select a T4 Colab runtime.")
    props = torch.cuda.get_device_properties(0)
    return {
        "timestamp_utc": utc_now(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "autoawq": package_version("autoawq"),
        "turboquant": package_version("turboquant"),
        "hqq": package_version("hqq"),
        "cuda_runtime": torch.version.cuda,
        "gpu_name": props.name,
        "gpu_total_mb": props.total_memory / 2**20,
        "compute_capability": f"{props.major}.{props.minor}",
        "seed": SEED,
        "turboquant_commit": TURBOQUANT_COMMIT,
        "contexts": CONTEXT_LENGTHS,
        "performance_blocks": PERFORMANCE_BLOCKS,
        "niah_items_per_condition": NIAH_ITEMS_PER_CONDITION,
        "baseline_dtype": "float16 (T4 does not support native bfloat16)",
    }

MANIFEST = gpu_manifest()
if "T4" not in MANIFEST["gpu_name"]:
    raise RuntimeError(f"Expected an NVIDIA T4, found {MANIFEST['gpu_name']!r}.")
if transformers.__version__ != "5.3.0":
    raise RuntimeError("This notebook is validated against transformers==5.3.0.")
(ROOT / "environment_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
display(pd.Series(MANIFEST, name="value"))

## 3. Atomic persistence and NVML power monitor

Energy is trapezoidally integrated from timestamped samples. CUDA is synchronized
immediately before monitor start and before monitor stop. Raw traces are retained.

In [ ]:
import pynvml

pynvml.nvmlInit()
NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)

def append_csv_atomic(path: Path, row: dict[str, Any], columns=RESULT_COLUMNS) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    lock = path.with_suffix(path.suffix + ".lock")
    while True:
        try:
            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.close(fd)
            break
        except FileExistsError:
            time.sleep(0.05)
    try:
        exists = path.exists() and path.stat().st_size > 0
        with path.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
            if not exists:
                writer.writeheader()
            writer.writerow({key: row.get(key) for key in columns})
            handle.flush()
            os.fsync(handle.fileno())
    finally:
        lock.unlink(missing_ok=True)

class GPUMonitor:
    def __init__(self, interval_s: float = POLL_INTERVAL_S):
        self.interval_s = interval_s
        self.samples: list[dict[str, float]] = []
        self._running = False
        self._thread: threading.Thread | None = None
        self._t0 = 0.0

    def _sample(self) -> None:
        try:
            memory = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
            util = pynvml.nvmlDeviceGetUtilizationRates(NVML_HANDLE)
            self.samples.append({
                "t_s": time.perf_counter() - self._t0,
                "power_w": pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0,
                "temperature_c": float(pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)),
                "gpu_util_pct": float(util.gpu),
                "memory_used_mb": memory.used / 2**20,
            })
        except pynvml.NVMLError as exc:
            self.samples.append({"t_s": time.perf_counter() - self._t0, "error": str(exc)})

    def _poll(self) -> None:
        deadline = self._t0
        while self._running:
            self._sample()
            deadline += self.interval_s
            time.sleep(max(0.0, deadline - time.perf_counter()))

    def start(self) -> None:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        self.samples = []
        self._running = True
        self._t0 = time.perf_counter()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self, idle_power_w: float) -> dict[str, float]:
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - self._t0
        self._running = False
        if self._thread:
            self._thread.join(timeout=2)
        self._sample()
        valid = [sample for sample in self.samples if "power_w" in sample]
        if len(valid) < 2:
            raise RuntimeError(f"Only {len(valid)} valid NVML power samples were collected.")
        times = np.asarray([sample["t_s"] for sample in valid], dtype=float)
        powers = np.asarray([sample["power_w"] for sample in valid], dtype=float)
        gross_j = float(np.trapezoid(powers, times))
        return {
            "monitor_elapsed_s": elapsed,
            "avg_power_w": float(gross_j / max(times[-1] - times[0], 1e-9)),
            "min_power_w": float(powers.min()),
            "max_power_w": float(powers.max()),
            "gross_energy_j": gross_j,
            "active_energy_j": max(0.0, gross_j - idle_power_w * (times[-1] - times[0])),
            "nvml_peak_used_mb": max(sample["memory_used_mb"] for sample in valid),
            "power_samples": len(valid),
        }

def measure_idle_power(seconds: float = 30.0) -> float:
    samples = []
    end = time.monotonic() + seconds
    while time.monotonic() < end:
        samples.append(pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0)
        time.sleep(POLL_INTERVAL_S)
    idle = float(np.median(samples))
    (ROOT / "idle_power.json").write_text(
        json.dumps({"timestamp_utc": utc_now(), "seconds": seconds, "median_w": idle, "samples_w": samples}, indent=2),
        encoding="utf-8",
    )
    return idle

IDLE_POWER_W = measure_idle_power(30.0)
print(f"Idle median: {IDLE_POWER_W:.2f} W")

## 4. Exact-token NIAH and reasoning inputs

Inputs are assembled directly as token IDs around a sentinel in Qwen's own chat
template. This avoids the historical notebook's approximate word-to-token conversion.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_BF16, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

SENTINEL = "<CSE407_INSERT_9f31>"
FILLER_TEXT = (
    "Archival note: the committee reviewed routine schedules, equipment inventories, "
    "weather summaries, and ordinary administrative records. No action was required. "
)
NEEDLE = "The secret project code name is SHADOW-8821."

def template_parts(instruction: str, enable_thinking: bool) -> tuple[list[int], list[int]]:
    messages = [
        {"role": "system", "content": "Follow the user's instruction exactly."},
        {"role": "user", "content": f"{instruction}\n\n{SENTINEL}"},
    ]
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    if rendered.count(SENTINEL) != 1:
        raise RuntimeError("The chat-template sentinel was not preserved exactly once.")
    left, right = rendered.split(SENTINEL)
    return tokenizer.encode(left, add_special_tokens=False), tokenizer.encode(right, add_special_tokens=False)

def repeated_ids(text: str, count: int) -> list[int]:
    unit = tokenizer.encode(text, add_special_tokens=False)
    if not unit:
        raise ValueError("Filler text tokenized to an empty sequence.")
    return (unit * math.ceil(count / len(unit)))[:count]

def make_niah_input(context_length: int, depth: float, item_id: int) -> dict[str, torch.Tensor]:
    instruction = (
        "Read the document. Complete the final statement using only the code name from the document. "
        "Return the code name and nothing else."
    )
    left, right = template_parts(instruction, enable_thinking=False)
    needle_ids = tokenizer.encode(f"\n{NEEDLE}\n", add_special_tokens=False)
    question_ids = tokenizer.encode("\nThe secret project code name is", add_special_tokens=False)
    available = context_length - len(left) - len(right) - len(needle_ids) - len(question_ids)
    if available < 32:
        raise ValueError(f"Context {context_length} is too short for fixed prompt overhead.")
    rng = random.Random(SEED * 100000 + context_length * 100 + item_id)
    jitter = rng.randint(-max(1, available // 100), max(1, available // 100))
    before = min(available, max(0, round(available * depth) + jitter))
    after = available - before
    ids = left + repeated_ids(FILLER_TEXT, before) + needle_ids + repeated_ids(FILLER_TEXT, after) + question_ids + right
    if len(ids) != context_length:
        raise AssertionError(f"Constructed {len(ids)} tokens, expected {context_length}.")
    return {
        "input_ids": torch.tensor([ids], dtype=torch.long),
        "attention_mask": torch.ones((1, context_length), dtype=torch.long),
    }

def reasoning_dataset(n: int = REASONING_ITEMS) -> list[dict[str, str]]:
    rng = random.Random(SEED)
    rows = []
    for index in range(n):
        a, b, c = rng.randint(12, 90), rng.randint(3, 20), rng.randint(2, 9)
        if index % 3 == 0:
            question, answer = f"What is ({a} + {b}) * {c}?", str((a + b) * c)
        elif index % 3 == 1:
            total = a * c + b
            question = f"A box has {total} cards. After {b} are removed, the rest are split equally among {c} people. How many does each person get?"
            answer = str(a)
        else:
            question = f"Sequence: {a}, {a+b}, {a+2*b}, {a+3*b}. What is the next number?"
            answer = str(a + 4 * b)
        rows.append({"item_id": f"reason_{index:02d}", "question": question, "answer": answer})
    return rows

def make_reasoning_input(item: dict[str, str], context_length: int = 8192) -> dict[str, torch.Tensor]:
    instruction = (
        "Ignore the archival notes, solve the final problem carefully, and put only the final integer in "
        "\\boxed{...}.\n\nArchival notes follow."
    )
    left, right = template_parts(instruction, enable_thinking=True)
    question_ids = tokenizer.encode(f"\n\nProblem: {item['question']}", add_special_tokens=False)
    filler_count = context_length - len(left) - len(right) - len(question_ids)
    if filler_count < 0:
        raise ValueError("Reasoning prompt overhead exceeds target context.")
    ids = left + repeated_ids(FILLER_TEXT, filler_count) + question_ids + right
    assert len(ids) == context_length
    return {
        "input_ids": torch.tensor([ids], dtype=torch.long),
        "attention_mask": torch.ones((1, context_length), dtype=torch.long),
    }

def niah_correct(response: str) -> bool:
    return "shadow-8821" in response.lower()

def extract_boxed_integer(response: str) -> str | None:
    boxed = re.findall(r"\\boxed\s*\{\s*(-?\d+)\s*\}", response)
    if boxed:
        return boxed[-1]
    integers = re.findall(r"(?<![\w.])-?\d+(?![\w.])", response)
    return integers[-1] if integers else None

for length in CONTEXT_LENGTHS:
    assert make_niah_input(length, 0.5, 0)["input_ids"].shape[1] == length
assert make_reasoning_input(reasoning_dataset(1)[0])["input_ids"].shape[1] == 8192
print("Exact-token prompt checks passed.")

## 5. Cache factories, physical storage accounting, and capability registry

In [ ]:
def tensor_storage_bytes(root: Any) -> int:
    """Count unique tensor storages reachable through Python attributes/containers."""
    seen_objects: set[int] = set()
    seen_storages: set[tuple[str, int]] = set()
    total = 0

    def visit(value: Any) -> None:
        nonlocal total
        object_id = id(value)
        if object_id in seen_objects:
            return
        seen_objects.add(object_id)
        if isinstance(value, torch.Tensor):
            if value.numel() == 0:
                return
            storage = value.untyped_storage()
            key = (str(value.device), storage.data_ptr())
            if key not in seen_storages:
                seen_storages.add(key)
                total += storage.nbytes()
            return
        if isinstance(value, dict):
            for child in value.values():
                visit(child)
        elif isinstance(value, (list, tuple, set)):
            for child in value:
                visit(child)
        elif hasattr(value, "__dict__"):
            for child in vars(value).values():
                visit(child)

    visit(root)
    return total

@dataclass(frozen=True)
class ExperimentConfig:
    code: str
    model_id: str
    requested_method: str
    cache_factory: Callable[[Any], Any] | None
    actual_method: str
    method_validated: bool
    blocked_reason: str = ""

def full_cache(model):
    return DynamicCache(config=model.config)

def hqq_cache(model):
    return QuantizedCache(
        backend="hqq",
        config=model.config,
        nbits=4,
        axis_key=1,
        axis_value=1,
        q_group_size=64,
        residual_length=128,
    )

def turboquant_cache(model):
    from turboquant import TurboQuantCache
    return TurboQuantCache(bits=4)

CONFIGS: dict[str, ExperimentConfig] = {
    "C0": ExperimentConfig("C0", MODEL_BF16, "FullKV", full_cache, "FullKV", True),
    "C1": ExperimentConfig("C1", MODEL_AWQ, "FullKV", full_cache, "FullKV", True),
    "C2": ExperimentConfig("C2", MODEL_AWQ, "TurboQuant-4bit", turboquant_cache, "TurboQuant-community-4bit", False),
    "C3": ExperimentConfig(
        "C3", MODEL_AWQ, "PyramidKV-1024", None, "BLOCKED", False,
        "Official KVCache-Factory has no Qwen3 adapter and pins transformers==4.44.2.",
    ),
    "C4": ExperimentConfig(
        "C4", MODEL_AWQ, "PyramidKV-1024+TurboQuant-4bit", None, "BLOCKED", False,
        "C4 is blocked until a Qwen3 PyramidKV/SnapKV adapter passes structural and quality pilots.",
    ),
}

def display_capabilities() -> pd.DataFrame:
    return pd.DataFrame([
        {
            "config": cfg.code,
            "model": cfg.model_id,
            "requested": cfg.requested_method,
            "actual": cfg.actual_method,
            "validated": cfg.method_validated,
            "status": "ready" if cfg.cache_factory else cfg.blocked_reason,
        }
        for cfg in CONFIGS.values()
    ])

display(display_capabilities())

## 6. Model loading and cache pilots

Do not run the full matrix until all requested active configurations pass. The AWQ
model is intentionally unfused because fused modules use a separate cache and cannot
be combined validly with KV-cache treatments.

In [ ]:
def clear_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_model(model_id: str):
    clear_cuda()
    kwargs = {
        "device_map": {"": 0},
        "attn_implementation": "sdpa",
        "low_cpu_mem_usage": True,
        "trust_remote_code": False,
    }
    if model_id == MODEL_BF16:
        kwargs["dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()
    if model.config.model_type != "qwen3":
        raise RuntimeError(f"Expected model_type='qwen3', got {model.config.model_type!r}.")
    if model.config.num_hidden_layers != 36 or model.config.num_key_value_heads != 8 or model.config.head_dim != 128:
        raise RuntimeError("Qwen3-4B architecture does not match the preregistered 36-layer, 8-KV-head, 128-dim design.")
    return model

def validate_full_cache(model) -> dict[str, Any]:
    inputs = make_niah_input(512, 0.5, 999)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    cache = full_cache(model)
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
    assert output.past_key_values is cache
    assert int(cache.get_seq_length()) == 512
    live = tensor_storage_bytes(cache)
    assert live > 0
    return {"actual_method": "FullKV", "cache_live_bytes": live, "seq_length": 512}

def validate_turboquant_or_fallback(model) -> dict[str, Any]:
    inputs = make_niah_input(512, 0.5, 998)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    errors = []
    try:
        cache = turboquant_cache(model)
        with torch.inference_mode():
            output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
        assert output.past_key_values is cache
        assert int(cache.get_seq_length()) == 512
        assert len(cache.layers) == model.config.num_hidden_layers
        layer = cache.layers[0]
        assert layer._key_indices.dtype == torch.uint8
        assert layer._key_indices.shape[-1] == model.config.head_dim // 2
        assert layer._value_indices.shape[-1] == model.config.head_dim // 2
        live = tensor_storage_bytes(cache)
        reported = cache.memory_usage_bytes()
        if live <= 0 or reported["compressed_bytes"] <= 0:
            raise AssertionError("TurboQuant did not expose non-empty compressed storage.")
        fp16_cache_bytes = (
            model.config.num_hidden_layers
            * model.config.num_key_value_heads
            * 512
            * model.config.head_dim
            * 2  # keys and values
            * 2  # FP16 bytes
        )
        if live >= fp16_cache_bytes:
            raise AssertionError(
                "TurboQuant compressed indices exist, but reachable live cache storage "
                f"({live / 2**20:.1f} MiB) is not below the FP16 equivalent "
                f"({fp16_cache_bytes / 2**20:.1f} MiB)."
            )
        return {
            "actual_method": "TurboQuant-community-4bit",
            "cache_factory": turboquant_cache,
            "cache_live_bytes": live,
            "reported": reported,
            "errors": errors,
        }
    except Exception as exc:
        errors.append(f"TurboQuant rejected: {type(exc).__name__}: {exc}")
        clear_cuda()

    cache = hqq_cache(model)
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
    assert output.past_key_values is cache
    assert int(cache.get_seq_length()) == 512
    live = tensor_storage_bytes(cache)
    if live <= 0:
        raise AssertionError("HQQ fallback did not expose cache storage.")
    fp16_cache_bytes = (
        model.config.num_hidden_layers
        * model.config.num_key_value_heads
        * 512
        * model.config.head_dim
        * 2
        * 2
    )
    if live >= fp16_cache_bytes:
        raise AssertionError("HQQ fallback did not reduce reachable storage below the FP16 cache.")
    return {
        "actual_method": "HF_HQQ_4BIT_KIVI_INSPIRED",
        "cache_factory": hqq_cache,
        "cache_live_bytes": live,
        "errors": errors,
    }

# Run pilots one model family at a time.
pilot_rows = []
bf16_model = load_model(MODEL_BF16)
pilot_rows.append({"config": "C0", **validate_full_cache(bf16_model)})
del bf16_model
clear_cuda()

awq_model = load_model(MODEL_AWQ)
pilot_rows.append({"config": "C1", **validate_full_cache(awq_model)})
c2_pilot = validate_turboquant_or_fallback(awq_model)
pilot_rows.append({key: value for key, value in {"config": "C2", **c2_pilot}.items() if key != "cache_factory"})
CONFIGS["C2"] = ExperimentConfig(
    "C2", MODEL_AWQ, "TurboQuant-4bit", c2_pilot["cache_factory"], c2_pilot["actual_method"], True,
    "; ".join(c2_pilot.get("errors", [])),
)
del awq_model
clear_cuda()

(ROOT / "cache_pilots.json").write_text(json.dumps(pilot_rows, indent=2, default=str), encoding="utf-8")
display(pd.DataFrame(pilot_rows))
display(display_capabilities())

## 7. Timed prefill and decode runner

Performance runs use fixed 128-token greedy decode. NIAH uses a separate short
completion workload. Model loading and prompt construction are outside the measured
interval. The runner records actual output-token counts rather than configured maxima.

In [ ]:
def wait_for_cooldown(target_c: int = COOLDOWN_TARGET_C, timeout_s: int = COOLDOWN_TIMEOUT_S) -> float:
    start = time.monotonic()
    while time.monotonic() - start < timeout_s:
        temperature = pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
        if temperature <= target_c:
            return float(temperature)
        time.sleep(2)
    return float(pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU))

def decode_response(token_ids: torch.Tensor) -> str:
    return tokenizer.decode(token_ids.tolist(), skip_special_tokens=True, clean_up_tokenization_spaces=False)

def measured_generate(
    model,
    cfg: ExperimentConfig,
    inputs_cpu: dict[str, torch.Tensor],
    max_new_tokens: int,
    trace_stem: str,
) -> dict[str, Any]:
    if cfg.cache_factory is None or not cfg.method_validated:
        raise RuntimeError(f"{cfg.code} is not executable: {cfg.blocked_reason}")
    wait_for_cooldown()
    clear_cuda()
    inputs = {key: value.to(model.device) for key, value in inputs_cpu.items()}
    cache = cfg.cache_factory(model)
    loaded_mb = torch.cuda.memory_allocated() / 2**20
    monitor = GPUMonitor()
    monitor.start()
    started = time.perf_counter()
    try:
        prefill_started = time.perf_counter()
        with torch.inference_mode():
            output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
        torch.cuda.synchronize()
        prefill_s = time.perf_counter() - prefill_started

        next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = []
        attention_mask = inputs["attention_mask"]
        decode_started = time.perf_counter()
        for _ in range(max_new_tokens):
            generated.append(next_token)
            if int(next_token.item()) in {tokenizer.eos_token_id, 151645}:
                break
            attention_mask = torch.cat(
                [attention_mask, torch.ones((attention_mask.shape[0], 1), dtype=attention_mask.dtype, device=model.device)],
                dim=1,
            )
            with torch.inference_mode():
                output = model(
                    input_ids=next_token,
                    attention_mask=attention_mask,
                    past_key_values=cache,
                    use_cache=True,
                    logits_to_keep=1,
                )
            next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        torch.cuda.synchronize()
        decode_s = time.perf_counter() - decode_started
        total_s = time.perf_counter() - started
        power = monitor.stop(IDLE_POWER_W)
    except Exception:
        with contextlib.suppress(Exception):
            monitor.stop(IDLE_POWER_W)
        raise

    output_ids = torch.cat(generated, dim=1)[0] if generated else torch.empty(0, dtype=torch.long, device=model.device)
    response = decode_response(output_ids.detach().cpu())
    cache_live_mb = tensor_storage_bytes(cache) / 2**20
    trace_path = TRACE_DIR / f"{trace_stem}.jsonl"
    with trace_path.open("w", encoding="utf-8") as handle:
        for sample in monitor.samples:
            handle.write(json.dumps(sample) + "\n")
    output_tokens = int(output_ids.numel())
    return {
        "response": response,
        "input_tokens": int(inputs["input_ids"].shape[1]),
        "output_tokens": output_tokens,
        "prefill_s": prefill_s,
        "decode_s": decode_s,
        "total_s": total_s,
        "prefill_tok_s": int(inputs["input_ids"].shape[1]) / max(prefill_s, 1e-9),
        "decode_tok_s": output_tokens / max(decode_s, 1e-9),
        "torch_loaded_mb": loaded_mb,
        "torch_peak_mb": torch.cuda.max_memory_allocated() / 2**20,
        "cache_live_mb": cache_live_mb,
        "trace_file": str(trace_path.relative_to(ROOT)),
        "energy_per_output_token_j": power["gross_energy_j"] / max(output_tokens, 1),
        **power,
    }

def base_result(cfg: ExperimentConfig, experiment: str, context: int, **extra) -> dict[str, Any]:
    return {
        "run_id": str(uuid.uuid4()),
        "timestamp_utc": utc_now(),
        "experiment": experiment,
        "config": cfg.code,
        "requested_method": cfg.requested_method,
        "actual_method": cfg.actual_method,
        "method_validated": cfg.method_validated,
        "model_id": cfg.model_id,
        "context_target": context,
        "idle_power_w": IDLE_POWER_W,
        "valid": False,
        "error": "",
        **extra,
    }

def run_and_save(model, cfg: ExperimentConfig, inputs, output_path: Path, trace_stem: str, **metadata):
    row = base_result(cfg, metadata.pop("experiment"), metadata.pop("context_target"), **metadata)
    try:
        row.update(measured_generate(model, cfg, inputs, metadata.get("max_new_tokens", DECODE_TOKENS), trace_stem))
        row["valid"] = (
            row["power_samples"] >= 2
            and 5.0 <= row["min_power_w"] <= row["max_power_w"] <= 100.0
            and row["input_tokens"] == row["context_target"]
        )
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()
    append_csv_atomic(output_path, row)
    return row

## 8. Mandatory sanity checks

C0 must retrieve the needle at 512 tokens. C1 should do the same. C2 must generate
finite output and must use the method selected by its pilot. Stop if these fail.

In [ ]:
SANITY_CSV = RAW_DIR / "sanity.csv"
sanity_rows = []
for config_code in ("C0", "C1", "C2"):
    cfg = CONFIGS[config_code]
    model = load_model(cfg.model_id)
    inputs = make_niah_input(512, 0.5, 10000 + len(sanity_rows))
    row = run_and_save(
        model, cfg, inputs, SANITY_CSV, f"sanity_{config_code}",
        experiment="sanity", context_target=512, block=-1, item_id="sanity",
        thinking=False, needle_depth=0.5, max_new_tokens=NIAH_MAX_NEW_TOKENS,
    )
    row["correct"] = niah_correct(row.get("response", ""))
    sanity_rows.append(row)
    del model
    clear_cuda()

display(pd.DataFrame(sanity_rows)[["config", "actual_method", "input_tokens", "output_tokens", "correct", "valid", "error"]])
if not sanity_rows[0]["correct"]:
    raise RuntimeError("C0 failed 512-token NIAH. Fix the prompt/evaluator before collecting data.")
if not all(row["valid"] for row in sanity_rows):
    raise RuntimeError("At least one sanity run failed measurement validation.")

## 9. Full performance matrix (restart-safe)

Only validated configurations run. C0 is collected in a BF16 model session. C1/C2
are randomized within each AWQ block to reduce order bias. C3/C4 remain in the
capability table as blocked rather than being replaced by misleading FullKV rows.

In [ ]:
PERF_CSV = RAW_DIR / "performance.csv"

def completed_keys(path: Path, experiment: str) -> set[tuple]:
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    frame = frame[(frame["experiment"] == experiment) & (frame["valid"] == True)]
    return set(zip(frame["config"], frame["context_target"], frame["block"], strict=False))

def run_performance_for_model(model_id: str, config_codes: list[str]) -> None:
    done = completed_keys(PERF_CSV, "performance")
    model = load_model(model_id)
    try:
        for block in range(PERFORMANCE_BLOCKS):
            order = list(config_codes)
            random.Random(SEED + block).shuffle(order)
            for context in CONTEXT_LENGTHS:
                inputs = make_niah_input(context, 0.5, 20000 + block)
                for code in order:
                    if (code, context, block) in done:
                        continue
                    cfg = CONFIGS[code]
                    row = run_and_save(
                        model, cfg, inputs, PERF_CSV, f"perf_{code}_{context}_b{block:02d}",
                        experiment="performance", context_target=context, block=block,
                        item_id=f"perf_{block:02d}", thinking=False, needle_depth=0.5,
                        max_new_tokens=DECODE_TOKENS,
                    )
                    print(code, context, block, row["valid"], row.get("gross_energy_j"), row.get("error", ""))
    finally:
        del model
        clear_cuda()

if RUN_FULL_PERFORMANCE:
    run_performance_for_model(MODEL_BF16, ["C0"])
    run_performance_for_model(MODEL_AWQ, ["C1", "C2"])
else:
    print("Performance matrix skipped. Set RUN_FULL_PERFORMANCE=True in Section 2 to collect it.")

## 10. NIAH retrieval matrix

Thirty matched items per condition are balanced over 25%, 50%, and 75% needle depth.

In [ ]:
NIAH_CSV = RAW_DIR / "niah.csv"

def completed_items(path: Path, experiment: str) -> set[tuple]:
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    frame = frame[(frame["experiment"] == experiment) & (frame["valid"] == True)]
    return set(zip(frame["config"], frame["context_target"], frame["item_id"], strict=False))

def run_niah_for_model(model_id: str, config_codes: list[str]) -> None:
    done = completed_items(RAW_DIR / "niah_scores.csv", "niah")
    model = load_model(model_id)
    depths = (0.25, 0.50, 0.75)
    try:
        for context in CONTEXT_LENGTHS:
            for item_index in range(NIAH_ITEMS_PER_CONDITION):
                item_id = f"niah_{item_index:02d}"
                depth = depths[item_index % len(depths)]
                inputs = make_niah_input(context, depth, item_index)
                order = list(config_codes)
                random.Random(SEED + context + item_index).shuffle(order)
                for code in order:
                    if (code, context, item_id) in done:
                        continue
                    cfg = CONFIGS[code]
                    row = run_and_save(
                        model, cfg, inputs, NIAH_CSV, f"niah_{code}_{context}_{item_index:02d}",
                        experiment="niah", context_target=context, block=item_index,
                        item_id=item_id, thinking=False, needle_depth=depth,
                        max_new_tokens=NIAH_MAX_NEW_TOKENS,
                    )
                    row["correct"] = niah_correct(row.get("response", ""))
                    # Persist scoring separately because the measurement row was already written.
                    append_csv_atomic(RAW_DIR / "niah_scores.csv", row)
                    print(code, context, item_id, row["correct"], row["valid"])
    finally:
        del model
        clear_cuda()

if RUN_FULL_NIAH:
    run_niah_for_model(MODEL_BF16, ["C0"])
    run_niah_for_model(MODEL_AWQ, ["C1", "C2"])
else:
    print("NIAH matrix skipped. Set RUN_FULL_NIAH=True in Section 2 to collect it.")

## 11. Thinking-mode exact-answer subset

C4 cannot run until eviction support is validated. The notebook records C0/C1 and
C2 as an informative quantized-cache extension; the capability table documents the
preregistered C4 omission.

In [ ]:
THINK_CSV = RAW_DIR / "thinking.csv"

def run_thinking_for_model(model_id: str, config_codes: list[str]) -> None:
    done = completed_items(RAW_DIR / "thinking_scores.csv", "thinking")
    model = load_model(model_id)
    items = reasoning_dataset()
    try:
        for item_index, item in enumerate(items):
            inputs = make_reasoning_input(item)
            order = list(config_codes)
            random.Random(SEED + item_index).shuffle(order)
            for code in order:
                if (code, 8192, item["item_id"]) in done:
                    continue
                cfg = CONFIGS[code]
                row = run_and_save(
                    model, cfg, inputs, THINK_CSV, f"think_{code}_{item_index:02d}",
                    experiment="thinking", context_target=8192, block=item_index,
                    item_id=item["item_id"], thinking=True, needle_depth=np.nan,
                    max_new_tokens=THINKING_MAX_NEW_TOKENS,
                )
                predicted = extract_boxed_integer(row.get("response", ""))
                row["correct"] = predicted == item["answer"]
                row["expected_answer"] = item["answer"]
                row["predicted_answer"] = predicted
                append_csv_atomic(
                    RAW_DIR / "thinking_scores.csv", row,
                    columns=RESULT_COLUMNS + ["expected_answer", "predicted_answer"],
                )
                print(code, item["item_id"], predicted, item["answer"], row["correct"], row["valid"])
    finally:
        del model
        clear_cuda()

if RUN_FULL_THINKING:
    run_thinking_for_model(MODEL_BF16, ["C0"])
    run_thinking_for_model(MODEL_AWQ, ["C1", "C2"])
else:
    print("Thinking subset skipped. Set RUN_FULL_THINKING=True in Section 2 to collect it.")

## 12. Optional active-cache perplexity

This computes next-token loss sequentially with the selected cache active. It is
intentionally separate from timed energy runs. Set `RUN_PERPLEXITY = True` after the
main pilots succeed. Results are descriptive because this small fixed sample is not a
full WikiText-2 benchmark.

In [ ]:
RUN_PERPLEXITY = False

def active_cache_perplexity(model, cfg: ExperimentConfig, token_ids: torch.Tensor, chunk_size: int = 128) -> float:
    cache = cfg.cache_factory(model)
    token_ids = token_ids.to(model.device)
    total_nll = 0.0
    total_targets = 0
    with torch.inference_mode():
        for start in range(0, token_ids.shape[1] - 1, chunk_size):
            stop = min(start + chunk_size, token_ids.shape[1] - 1)
            current = token_ids[:, start:stop]
            targets = token_ids[:, start + 1:stop + 1]
            output = model(input_ids=current, past_key_values=cache, use_cache=True, logits_to_keep=0)
            logits = output.logits[:, -targets.shape[1]:, :].float()
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.shape[-1]), targets.reshape(-1), reduction="sum"
            )
            total_nll += float(loss.item())
            total_targets += targets.numel()
    return math.exp(total_nll / total_targets)

if RUN_PERPLEXITY:
    sample = repeated_ids(
        "Energy-efficient computing evaluates useful work relative to measured resource consumption. ",
        2049,
    )
    tokens = torch.tensor([sample], dtype=torch.long)
    ppl_rows = []
    for model_id, codes in ((MODEL_BF16, ["C0"]), (MODEL_AWQ, ["C1", "C2"])):
        model = load_model(model_id)
        for code in codes:
            ppl_rows.append({"config": code, "actual_method": CONFIGS[code].actual_method, "perplexity": active_cache_perplexity(model, CONFIGS[code], tokens)})
        del model
        clear_cuda()
    pd.DataFrame(ppl_rows).to_csv(RAW_DIR / "perplexity.csv", index=False)
    display(pd.DataFrame(ppl_rows))

## 13. Validation, statistics, confidence intervals, and figures

Performance uses blocks as paired units. Friedman and paired Wilcoxon tests are run
only when at least three validated configurations have complete paired blocks;
otherwise pairwise Wilcoxon is reported. Bonferroni correction uses the number of
actually tested pairs. Binary matched outcomes use McNemar tests.

In [ ]:
def load_valid(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    if "valid" in frame:
        frame = frame[frame["valid"].astype(str).str.lower().isin(["true", "1"])]
    return frame

perf = load_valid(PERF_CSV)
niah = load_valid(RAW_DIR / "niah_scores.csv")
thinking = load_valid(RAW_DIR / "thinking_scores.csv")

def mean_ci(values, confidence=0.95):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(values) < 2:
        return (float(values.mean()) if len(values) else np.nan, np.nan, np.nan)
    mean = float(values.mean())
    sem = st.sem(values)
    low, high = st.t.interval(confidence, len(values) - 1, loc=mean, scale=sem)
    return mean, float(low), float(high)

def binary_ci(successes: int, trials: int, confidence=0.95):
    if trials == 0:
        return np.nan, np.nan
    result = binomtest(successes, trials).proportion_ci(confidence_level=confidence, method="exact")
    return float(result.low), float(result.high)

summary_rows = []
if not perf.empty:
    for (config, actual, context), group in perf.groupby(["config", "actual_method", "context_target"]):
        energy_mean, energy_low, energy_high = mean_ci(group["gross_energy_j"])
        summary_rows.append({
            "config": config, "actual_method": actual, "context": context, "n": len(group),
            "energy_mean_j": energy_mean, "energy_ci_low": energy_low, "energy_ci_high": energy_high,
            "power_mean_w": group["avg_power_w"].mean(), "peak_vram_mean_mb": group["torch_peak_mb"].mean(),
            "decode_tok_s_mean": group["decode_tok_s"].mean(), "cache_live_mean_mb": group["cache_live_mb"].mean(),
        })
perf_summary = pd.DataFrame(summary_rows)
perf_summary.to_csv(ROOT / "performance_summary.csv", index=False)

accuracy_rows = []
for name, frame in (("niah", niah), ("thinking", thinking)):
    if frame.empty:
        continue
    frame["correct"] = frame["correct"].astype(str).str.lower().isin(["true", "1"])
    for keys, group in frame.groupby(["config", "actual_method", "context_target"]):
        successes = int(group["correct"].sum())
        low, high = binary_ci(successes, len(group))
        accuracy_rows.append({
            "experiment": name, "config": keys[0], "actual_method": keys[1], "context": keys[2],
            "successes": successes, "n": len(group), "accuracy": successes / len(group),
            "ci_low": low, "ci_high": high,
        })
accuracy_summary = pd.DataFrame(accuracy_rows)
accuracy_summary.to_csv(ROOT / "accuracy_summary.csv", index=False)

def paired_performance_tests(frame: pd.DataFrame, metric="gross_energy_j") -> pd.DataFrame:
    rows = []
    for context, context_frame in frame.groupby("context_target"):
        pivot = context_frame.pivot_table(index="block", columns="config", values=metric, aggfunc="first").dropna()
        configs = list(pivot.columns)
        pairs = [(configs[i], configs[j]) for i in range(len(configs)) for j in range(i + 1, len(configs))]
        for first, second in pairs:
            differences = pivot[first] - pivot[second]
            if len(differences) < 5 or np.allclose(differences, 0):
                statistic, p_value = np.nan, np.nan
            else:
                statistic, p_value = wilcoxon(pivot[first], pivot[second], alternative="two-sided")
            effect_r = np.nan
            if pd.notna(p_value) and p_value > 0:
                z = st.norm.isf(p_value / 2)
                effect_r = float(z / math.sqrt(len(differences)))
            rows.append({
                "context": context, "metric": metric, "config_a": first, "config_b": second,
                "n_pairs": len(differences), "wilcoxon_w": statistic, "p_raw": p_value,
                "median_difference": differences.median(), "effect_r_abs": effect_r,
            })
    result = pd.DataFrame(rows)
    if not result.empty:
        valid_tests = result["p_raw"].notna().sum()
        result["p_bonferroni"] = np.minimum(result["p_raw"] * max(valid_tests, 1), 1.0)
    return result

performance_tests = paired_performance_tests(perf) if not perf.empty else pd.DataFrame()
performance_tests.to_csv(ROOT / "performance_pairwise_tests.csv", index=False)

def performance_omnibus_tests(frame: pd.DataFrame, metric="gross_energy_j") -> pd.DataFrame:
    rows = []
    for context, context_frame in frame.groupby("context_target"):
        pivot = context_frame.pivot_table(index="block", columns="config", values=metric, aggfunc="first").dropna()
        if pivot.shape[1] >= 3 and pivot.shape[0] >= 5:
            statistic, p_value = friedmanchisquare(*(pivot[column].to_numpy() for column in pivot.columns))
            kendall_w = statistic / (pivot.shape[0] * (pivot.shape[1] - 1))
            rows.append({
                "context": context, "metric": metric, "test": "Friedman",
                "n_blocks": pivot.shape[0], "n_configs": pivot.shape[1],
                "statistic": statistic, "p_value": p_value, "kendall_w": kendall_w,
            })
    return pd.DataFrame(rows)

performance_omnibus = performance_omnibus_tests(perf) if not perf.empty else pd.DataFrame()
performance_omnibus.to_csv(ROOT / "performance_omnibus_tests.csv", index=False)

def matched_binary_tests(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if frame.empty:
        return pd.DataFrame()
    frame = frame.copy()
    frame["correct"] = frame["correct"].astype(str).str.lower().isin(["true", "1"])
    for context, context_frame in frame.groupby("context_target"):
        pivot = context_frame.pivot_table(index="item_id", columns="config", values="correct", aggfunc="first").dropna()
        configs = list(pivot.columns)
        for i in range(len(configs)):
            for j in range(i + 1, len(configs)):
                first, second = configs[i], configs[j]
                table = pd.crosstab(pivot[first], pivot[second]).reindex(index=[False, True], columns=[False, True], fill_value=0)
                result = mcnemar(table.to_numpy(), exact=True)
                rows.append({
                    "context": context, "config_a": first, "config_b": second, "n_pairs": len(pivot),
                    "a_wrong_b_right": int(table.loc[False, True]), "a_right_b_wrong": int(table.loc[True, False]),
                    "statistic": result.statistic, "p_raw": result.pvalue,
                })
    output = pd.DataFrame(rows)
    if not output.empty:
        output["p_bonferroni"] = np.minimum(output["p_raw"] * len(output), 1.0)
    return output

niah_tests = matched_binary_tests(niah)
thinking_tests = matched_binary_tests(thinking)
niah_tests.to_csv(ROOT / "niah_mcnemar_tests.csv", index=False)
thinking_tests.to_csv(ROOT / "thinking_mcnemar_tests.csv", index=False)

display(perf_summary)
display(accuracy_summary)
display(performance_omnibus)
display(performance_tests)

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
if not perf_summary.empty:
    fig, axes = plt.subplots(2, 2, figsize=(18, 13))
    sns.lineplot(data=perf_summary, x="context", y="energy_mean_j", hue="config", marker="o", ax=axes[0, 0])
    axes[0, 0].set(title="Gross GPU Energy", ylabel="Energy (J)", xlabel="Input tokens")
    sns.lineplot(data=perf_summary, x="context", y="peak_vram_mean_mb", hue="config", marker="o", ax=axes[0, 1])
    axes[0, 1].set(title="Peak Allocated VRAM", ylabel="MiB", xlabel="Input tokens")
    sns.lineplot(data=perf_summary, x="context", y="decode_tok_s_mean", hue="config", marker="o", ax=axes[1, 0])
    axes[1, 0].set(title="Greedy Decode Throughput", ylabel="tokens/s", xlabel="Input tokens")
    sns.lineplot(data=perf_summary, x="context", y="cache_live_mean_mb", hue="config", marker="o", ax=axes[1, 1])
    axes[1, 1].set(title="Reachable Live Cache Storage", ylabel="MiB", xlabel="Input tokens")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "performance_overview.png", dpi=200, bbox_inches="tight")
    plt.show()

if not accuracy_summary.empty:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=accuracy_summary, x="context", y="accuracy", hue="config")
    plt.ylim(0, 1.05)
    plt.title("Exact Accuracy by Context and Configuration")
    plt.ylabel("Accuracy")
    plt.xlabel("Input tokens")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "accuracy.png", dpi=200, bbox_inches="tight")
    plt.show()

# Pareto frontier at 8192: a point is dominated if another has no higher energy and no lower accuracy.
if not perf_summary.empty and not niah.empty:
    energy_8k = perf_summary[perf_summary["context"] == 8192][["config", "energy_mean_j"]]
    acc_8k = accuracy_summary[(accuracy_summary["experiment"] == "niah") & (accuracy_summary["context"] == 8192)][["config", "accuracy"]]
    pareto = energy_8k.merge(acc_8k, on="config")
    pareto["pareto"] = True
    for index, row in pareto.iterrows():
        dominated = ((pareto["energy_mean_j"] <= row["energy_mean_j"]) & (pareto["accuracy"] >= row["accuracy"]) & ((pareto["energy_mean_j"] < row["energy_mean_j"]) | (pareto["accuracy"] > row["accuracy"]))).any()
        pareto.loc[index, "pareto"] = not dominated
    pareto.to_csv(ROOT / "pareto_8192.csv", index=False)
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=pareto, x="energy_mean_j", y="accuracy", hue="config", style="pareto", s=180)
    for _, row in pareto.iterrows():
        plt.annotate(row["config"], (row["energy_mean_j"], row["accuracy"]), xytext=(6, 6), textcoords="offset points")
    plt.title("8192-token Energy-Accuracy Pareto Analysis")
    plt.xlabel("Mean gross GPU energy (J), lower is better")
    plt.ylabel("NIAH accuracy, higher is better")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "pareto_8192.png", dpi=200, bbox_inches="tight")
    plt.show()

## 14. Completeness and provenance report

In [ ]:
def completeness_report() -> pd.DataFrame:
    rows = []
    expected_perf = PERFORMANCE_BLOCKS
    expected_niah = NIAH_ITEMS_PER_CONDITION
    for code, cfg in CONFIGS.items():
        for context in CONTEXT_LENGTHS:
            perf_n = len(perf[(perf["config"] == code) & (perf["context_target"] == context)]) if not perf.empty else 0
            niah_n = len(niah[(niah["config"] == code) & (niah["context_target"] == context)]) if not niah.empty else 0
            rows.append({
                "config": code, "context": context, "requested_method": cfg.requested_method,
                "actual_method": cfg.actual_method, "validated": cfg.method_validated,
                "performance_valid": perf_n, "performance_expected": expected_perf,
                "niah_valid": niah_n, "niah_expected": expected_niah,
                "blocked_reason": cfg.blocked_reason,
            })
    return pd.DataFrame(rows)

completeness = completeness_report()
completeness.to_csv(ROOT / "completeness.csv", index=False)
display(completeness)

provenance = {
    "generated_utc": utc_now(),
    "environment": MANIFEST,
    "idle_power_w": IDLE_POWER_W,
    "configs": {code: asdict(cfg) | {"cache_factory": getattr(cfg.cache_factory, "__name__", None)} for code, cfg in CONFIGS.items()},
    "limitations": [
        "Official PyramidKV and SnapKV do not support Qwen3; C3/C4 are blocked rather than imitated.",
        "TurboQuant is a community implementation, not an official artifact from the paper authors.",
        "GPU-only NVML energy excludes host CPU, cooling, networking, and embodied carbon.",
        "Ten performance blocks are below the course's aspirational n>=30 and limit statistical power.",
        "HQQ fallback is KIVI-inspired but is not the original KIVI implementation.",
    ],
}
(ROOT / "experiment_provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")
print(f"All artifacts saved under: {ROOT}")